In [1]:
import pandas as pd
from common import evaluate

df = pd.read_csv("data/reference.csv", keep_default_na=False, na_values=[""])
print(df.to_string(max_rows=8))

      step    lat     lon     pmin_hpa    vmax_ms    vmax_kt                 init            sid   name basin                valid  bt_lat  bt_lon  bt_wind_kt  bt_pres_hpa  track_err_km  aifs0_vmax   aifs0_pmin  aifs_dvmax  aifs_vmax_runmax  aifs_pmin_runmin  aifs_vmax_prev6  aifs_dvmax_6h  speed_kmh  obs0_wind  obs0_pres  obs_dwind_12h  obs_dwind_24h  age_h  init_bias  persist_vmax_kt   doy_sin   doy_cos  is_ep  aifs_v2  ri      res_v  corr_vmax_kt  erm_vmax_kt  eta_vmax_kt  eta1980_vmax_kt
0        6  13.50 -105.50  1007.638611  12.805787  24.892452  2025-05-29 00:00:00  2025149N12256  ALVIN    EP  2025-05-29 06:00:00    13.0  -105.6        30.0       1005.0     56.641194   25.391666  1006.245178   -0.499214         25.391666       1006.245178        25.391666      -0.499214  30.927976       30.0     1006.0            NaN            NaN    6.0  -4.608334        29.500786  0.546711 -0.837321      1        0   0   5.107548     34.157588    41.805071    40.978710        38.958485
1     

In [2]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, d_in, hidden):
        super().__init__()
        self.l1 = nn.Linear(d_in, hidden)
        self.l2 = nn.Linear(hidden, hidden)
        self.l3 = nn.Linear(hidden, 1)

    def forward(self, x):
        # pass x through Linear -> ReLU etc
        x = torch.relu(self.l1(x))
        x = torch.relu(self.l2(x))
        x = self.l3(x).squeeze(-1)
        return x


In [3]:
def grad_norm(loss, model):
    grads = torch.autograd.grad(loss, list(model.parameters()),
                                retain_graph=True, allow_unused=True)
    return torch.sqrt(sum((g ** 2).sum() for g in grads if g is not None))

In [4]:
from common import FEATURES
from sklearn.model_selection import GroupKFold
import numpy as np

X = df[FEATURES]
y = df["res_v"]
groups = df["sid"]
gkf = GroupKFold(n_splits=5)
n_seeds = 3
levels = torch.linspace(0.95, 0.999, 25)
pred = np.zeros(len(df))

for train_idx, test_idx in gkf.split(X, y, groups):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_test  = X.iloc[test_idx]

    med = X_train.median()
    X_train = X_train.fillna(med)
    X_test = X_test.fillna(med)

    mu, sd = X_train.mean(), X_train.std()
    X_train = (X_train - mu) / sd
    X_test = (X_test - mu) / sd

    X_train = torch.as_tensor(X_train.values).float()
    y_train = torch.as_tensor(y_train.values).float()
    X_test = torch.as_tensor(X_test.values).float()

    for seed in range(n_seeds):
        lam = 0.0
        torch.manual_seed(seed)
        model = MLP(d_in=22, hidden=64)          
        opt = torch.optim.Adam(model.parameters(), lr=3e-3)

        vmax_train = torch.as_tensor(X.iloc[train_idx]["vmax_kt"].values).float()
        bt_wind_kt = torch.as_tensor(df["bt_wind_kt"].iloc[train_idx].values).float()
        ref_q = torch.quantile(bt_wind_kt, levels)

        for it in range(600):
            opt.zero_grad()
            out = model(X_train)        # forward pass on Xtr

            
            corr = vmax_train + out
            pred_q = torch.quantile(corr, levels)
            
            penalty = abs(pred_q - ref_q).mean()

            mse = ((out - y_train) ** 2).mean() # plain MSE

            if it % 100 == 0:
                lam = float(grad_norm(mse, model) / (grad_norm(penalty, model) + 1e-8))

            loss =  mse + lam * penalty      
            loss.backward()
            opt.step()
            if it % 100 == 0:
                print("Iter: ", it)
                print("Loss: ", loss.item())
                print("MSE: ", mse.item())
                print("Lambda: ", lam)
                print("Penalty: ", penalty.item())


        with torch.no_grad():
            fold_pred = model(X_test).numpy()
            pred[test_idx] += fold_pred / n_seeds

/var/folders/kb/_6x8b1kj1k55mr8hzpd32rbr0000gn/T/ipykernel_4999/3395906042.py:27: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  y_train = torch.as_tensor(y_train.values).float()


Iter:  0
Loss:  4763.46923828125
MSE:  1548.633056640625
Lambda:  41.86651611328125
Penalty:  76.78776550292969
Iter:  100
Loss:  575.8883056640625
MSE:  533.2186279296875
Lambda:  27.720762252807617
Penalty:  1.539267897605896
Iter:  200
Loss:  417.290283203125
MSE:  398.2326965332031
Lambda:  13.942669868469238
Penalty:  1.3668545484542847
Iter:  300
Loss:  337.66937255859375
MSE:  315.89886474609375
Lambda:  14.727252960205078
Penalty:  1.4782474040985107
Iter:  400
Loss:  276.4395751953125
MSE:  263.3786315917969
Lambda:  9.239354133605957
Penalty:  1.4136223793029785
Iter:  500
Loss:  229.41969299316406
MSE:  220.0197296142578
Lambda:  6.7857279777526855
Penalty:  1.3852545022964478
Iter:  0
Loss:  4675.08154296875
MSE:  1553.3897705078125
Lambda:  40.59426498413086
Penalty:  76.89982604980469
Iter:  100
Loss:  556.5372924804688
MSE:  507.4273986816406
Lambda:  29.249103546142578
Penalty:  1.6790213584899902
Iter:  200
Loss:  451.0367736816406
MSE:  387.67364501953125
Lambda:  47.

In [ ]:
df["my_eta_kt"] = df["vmax_kt"] + pred
evaluate(df, "my_eta_kt")

(np.float64(13.047855038529761),
 np.float64(30.880538700900093),
 step
 6       8.370526
 12      9.291583
 18     10.152458
 24     10.905080
 30     11.565280
 36     11.910654
 42     12.329520
 48     12.907431
 54     13.186162
 60     13.438360
 66     13.993240
 72     13.917727
 78     14.482763
 84     14.643887
 90     14.863455
 96     15.175622
 102    15.752231
 108    15.649810
 114    15.670833
 120    15.818665
 126    16.108943
 132    16.192439
 138    16.278818
 144    16.466390
 150    16.254427
 156    17.151971
 162    16.597126
 168    17.218761
 dtype: float64)

In [ ]:
from common import tail_quantiles

tail_quantiles(df, ["bt_wind_kt", "my_eta_kt"])

,bt_wind_kt,my_mlp_kt
0.90,110.0,99.002976
0.95,120.0,118.319992
0.99,140.0,144.491423
